# 01 — Feature Store de Simulações para Modelo `fl_proposta`

Objetivo: criar uma primeira camada de features **sem leakage** para o modelo:

> Dada uma simulação, qual a probabilidade de virar proposta?

Esta versão usa apenas informações disponíveis no momento da simulação e cria features referentes a:
- atributos diretos da simulação;
- tempo;
- valor financiado, entrada, prazo, veículo e taxa;
- quantidade de simulações por cliente;
- dinâmica da jornada de simulação;
- estatísticas históricas de preço usando apenas passado.

> Observação: identificadores como `id_cpf_cnpj`, `id_pricing` e `id_chave_execucao` são mantidos para chaveamento e agregações, mas não devem ser usados diretamente como features no modelo.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 220)

RANDOM_STATE = 42

# Ajuste os caminhos conforme seu projeto
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
FEATURE_DIR = BASE_DIR / "data" / "features"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)


## 1. Carregamento da base

Este notebook aceita dois modos:

1. Usar um DataFrame já existente chamado `df_proposta` no ambiente.
2. Carregar um arquivo Parquet informado em `INPUT_PATH`.

A base esperada é a base de simulações para o primeiro modelo, contendo:
- `fl_proposta` com valores 0/1;
- colunas pós-proposta removidas;
- campos `_pond` removidos, caso só existam após a proposta.


In [ ]:
# Opção A: se df_proposta já existe no notebook, ele será usado.
# Opção B: caso contrário, informe o caminho do parquet abaixo.

INPUT_PATH = PROCESSED_DIR / "df_proposta_sem_leakage.parquet"

if "df_proposta" in globals():
    df = df_proposta.copy()
    print("Usando DataFrame existente: df_proposta")
else:
    if not INPUT_PATH.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {INPUT_PATH}. "
            "Defina INPUT_PATH corretamente ou crie df_proposta antes de rodar esta célula."
        )
    df = pd.read_parquet(INPUT_PATH)
    print(f"Base carregada de: {INPUT_PATH}")

print(df.shape)
df.head()


## 2. Validações básicas anti-leakage

Para o modelo de proposta, o target é `fl_proposta`.

Colunas como `fl_aprovado`, `fl_contrato`, `id_contrato`, `dt_contrato` e campos `_pond` não devem ser features deste modelo.


In [ ]:
required_cols = [
    "id_chave_execucao",
    "id_pricing",
    "id_cpf_cnpj",
    "id_cluster",
    "dt_simulacao",
    "tp_politica",
    "tp_teste_elasticidade",
    "fl_proposta",
    "num_prazo",
    "cod_rev",
    "vl_financiado",
    "vl_entrada",
    "pct_entrada",
    "tp_gh",
    "fl_zero",
    "num_ano_mod_veic",
    "tx_simul",
    "tp_retorno_comissao",
    "fx_ano_mod_veic",
    "fx_entrada",
    "periodo_arquivo",
]

missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Colunas obrigatórias ausentes: {missing_required}")

# Colunas que não devem existir ou não devem entrar no modelo 1
leakage_like_cols = [
    c for c in df.columns
    if (
        c.endswith("_pond")
        or c in [
            "fl_aprovado",
            "fl_contrato",
            "id_contrato",
            "dt_contrato",
            "dt_aprovacao",
            "id_aprovacao",
        ]
    )
]

print("Colunas suspeitas de leakage encontradas:", leakage_like_cols)

# Se existirem, não vamos usar como features.
df["dt_simulacao"] = pd.to_datetime(df["dt_simulacao"], errors="coerce")

assert set(df["fl_proposta"].dropna().unique()).issubset({0, 1}), "fl_proposta precisa estar em {0,1}"
print(df["fl_proposta"].value_counts(dropna=False))


## 3. Funções auxiliares


In [ ]:
def safe_divide(a, b):
    return np.where((b == 0) | pd.isna(b), np.nan, a / b)

def add_missing_flags(df, cols):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[f"fsim_is_missing__{col}"] = df[col].isna().astype("int8")
    df["fsim_qtd_missing_base"] = df[[c for c in cols if c in df.columns]].isna().sum(axis=1).astype("int16")
    return df

def make_interaction(df, col_a, col_b, new_col):
    df[new_col] = (
        df[col_a].astype("string").fillna("MISSING")
        + "__"
        + df[col_b].astype("string").fillna("MISSING")
    )
    return df


## 4. Features diretas e temporais da simulação

Essas features usam apenas a própria linha da simulação.


In [ ]:
df_feat = df.copy()

# Target
df_feat["target_proposta"] = df_feat["fl_proposta"].astype("int8")

# Datas
df_feat["fsim_ano_simulacao"] = df_feat["dt_simulacao"].dt.year.astype("Int16")
df_feat["fsim_mes_simulacao"] = df_feat["dt_simulacao"].dt.month.astype("Int8")
df_feat["fsim_dia_mes_simulacao"] = df_feat["dt_simulacao"].dt.day.astype("Int8")
df_feat["fsim_dia_semana_simulacao"] = df_feat["dt_simulacao"].dt.dayofweek.astype("Int8")
df_feat["fsim_semana_ano_simulacao"] = df_feat["dt_simulacao"].dt.isocalendar().week.astype("Int16")
df_feat["fsim_trimestre_simulacao"] = df_feat["dt_simulacao"].dt.quarter.astype("Int8")
df_feat["fsim_hora_simulacao"] = df_feat["dt_simulacao"].dt.hour.astype("Int8")
df_feat["fsim_is_inicio_mes"] = (df_feat["dt_simulacao"].dt.day <= 5).astype("int8")
df_feat["fsim_is_fim_mes"] = (df_feat["dt_simulacao"].dt.day >= 25).astype("int8")
df_feat["fsim_data_simulacao"] = df_feat["dt_simulacao"].dt.date.astype("string")
df_feat["fsim_ano_mes_simulacao"] = df_feat["dt_simulacao"].dt.strftime("%Y%m")

# Valores financeiros
df_feat["fsim_log_vl_financiado"] = np.log1p(df_feat["vl_financiado"].clip(lower=0))
df_feat["fsim_log_vl_entrada"] = np.log1p(df_feat["vl_entrada"].clip(lower=0))

df_feat["fsim_pct_entrada_calc"] = safe_divide(df_feat["vl_entrada"], df_feat["vl_financiado"])
df_feat["fsim_gap_pct_entrada"] = df_feat["pct_entrada"] - df_feat["fsim_pct_entrada_calc"]

df_feat["fsim_vl_financiado_por_prazo"] = safe_divide(df_feat["vl_financiado"], df_feat["num_prazo"])
df_feat["fsim_vl_entrada_por_prazo"] = safe_divide(df_feat["vl_entrada"], df_feat["num_prazo"])

# Veículo
df_feat["fsim_idade_veiculo"] = df_feat["fsim_ano_simulacao"] - df_feat["num_ano_mod_veic"]
df_feat["fsim_is_veiculo_0km_proxy"] = (df_feat["fsim_idade_veiculo"] <= 0).astype("int8")
df_feat["fsim_is_veiculo_9mais_anos_proxy"] = (df_feat["fsim_idade_veiculo"] >= 9).astype("int8")

# Taxa
df_feat["fsim_tx_x_log_vl_financiado"] = df_feat["tx_simul"] * df_feat["fsim_log_vl_financiado"]
df_feat["fsim_tx_por_prazo"] = safe_divide(df_feat["tx_simul"], df_feat["num_prazo"])
df_feat["fsim_tx_x_pct_entrada"] = df_feat["tx_simul"] * df_feat["pct_entrada"]

# Flags de qualidade/missing
cols_missing_check = [
    "id_cluster",
    "tp_politica",
    "tp_teste_elasticidade",
    "num_prazo",
    "cod_rev",
    "vl_financiado",
    "vl_entrada",
    "pct_entrada",
    "tp_gh",
    "fl_zero",
    "num_ano_mod_veic",
    "tx_simul",
    "tp_retorno_comissao",
    "fx_ano_mod_veic",
    "fx_entrada",
]
df_feat = add_missing_flags(df_feat, cols_missing_check)

df_feat[[
    "target_proposta",
    "fsim_ano_mes_simulacao",
    "fsim_log_vl_financiado",
    "fsim_pct_entrada_calc",
    "fsim_idade_veiculo",
    "fsim_tx_x_log_vl_financiado",
]].head()


## 5. Interações categóricas seguras

Essas features combinam características conhecidas no momento da simulação.


In [ ]:
interaction_pairs = [
    ("tp_gh", "fx_entrada", "fsim_inter_tp_gh__fx_entrada"),
    ("tp_gh", "fx_ano_mod_veic", "fsim_inter_tp_gh__fx_ano_mod_veic"),
    ("id_cluster", "fx_entrada", "fsim_inter_cluster__fx_entrada"),
    ("id_cluster", "fx_ano_mod_veic", "fsim_inter_cluster__fx_ano_mod_veic"),
    ("tp_politica", "fx_entrada", "fsim_inter_politica__fx_entrada"),
    ("tp_politica", "tp_gh", "fsim_inter_politica__tp_gh"),
    ("cod_rev", "tp_gh", "fsim_inter_revenda__tp_gh"),
    ("tp_teste_elasticidade", "tp_gh", "fsim_inter_teste_elasticidade__tp_gh"),
]

for a, b, new_col in interaction_pairs:
    df_feat = make_interaction(df_feat, a, b, new_col)

df_feat[[p[2] for p in interaction_pairs]].head()


## 6. Features de quantidade de simulações por cliente

Essas features capturam se o cliente está fazendo múltiplas simulações.

Regra anti-leakage:
- usar apenas contagens **até a simulação atual**;
- não usar `qtd_total_simulacoes_cliente` calculada olhando o futuro.


In [ ]:
df_feat = df_feat.sort_values(["id_cpf_cnpj", "dt_simulacao", "id_chave_execucao"]).copy()

# Quantidade acumulada por cliente até a linha atual
df_feat["fsim_cli_ordem_simulacao_ate_agora"] = (
    df_feat.groupby("id_cpf_cnpj").cumcount() + 1
).astype("int32")

df_feat["fsim_cli_is_primeira_simulacao"] = (
    df_feat["fsim_cli_ordem_simulacao_ate_agora"] == 1
).astype("int8")

df_feat["fsim_cli_is_terceira_ou_mais_simulacao"] = (
    df_feat["fsim_cli_ordem_simulacao_ate_agora"] >= 3
).astype("int8")

df_feat["fsim_cli_is_quinta_ou_mais_simulacao"] = (
    df_feat["fsim_cli_ordem_simulacao_ate_agora"] >= 5
).astype("int8")

# Quantidade por cliente no mesmo dia até a linha atual
df_feat["fsim_cli_qtd_simulacoes_dia_ate_agora"] = (
    df_feat.groupby(["id_cpf_cnpj", "fsim_data_simulacao"]).cumcount() + 1
).astype("int32")

# Datas anteriores por cliente
df_feat["fsim_cli_dt_primeira_simulacao"] = (
    df_feat.groupby("id_cpf_cnpj")["dt_simulacao"].transform("first")
)

df_feat["fsim_cli_dt_simulacao_anterior"] = (
    df_feat.groupby("id_cpf_cnpj")["dt_simulacao"].shift(1)
)

df_feat["fsim_cli_min_desde_primeira_simulacao"] = (
    (df_feat["dt_simulacao"] - df_feat["fsim_cli_dt_primeira_simulacao"]).dt.total_seconds() / 60
)

df_feat["fsim_cli_min_desde_simulacao_anterior"] = (
    (df_feat["dt_simulacao"] - df_feat["fsim_cli_dt_simulacao_anterior"]).dt.total_seconds() / 60
)

df_feat["fsim_cli_is_multiplas_simulacoes_mesmo_dia"] = (
    df_feat["fsim_cli_qtd_simulacoes_dia_ate_agora"] >= 2
).astype("int8")

df_feat[[
    "id_cpf_cnpj",
    "dt_simulacao",
    "fsim_cli_ordem_simulacao_ate_agora",
    "fsim_cli_qtd_simulacoes_dia_ate_agora",
    "fsim_cli_min_desde_simulacao_anterior",
]].head(10)


## 7. Features de jornada de simulação

Caso não exista uma chave oficial de jornada, este notebook cria uma proxy:

```text
id_jornada_proxy = cliente + revenda + data da simulação
```

Essa proxy considera que simulações do mesmo cliente, na mesma revenda, no mesmo dia, fazem parte da mesma negociação.

Ajuste essa regra se você tiver uma chave melhor, como `id_negociacao`, `id_sessao`, `id_veiculo` ou outra.


In [ ]:
# Proxy simples de jornada
df_feat["id_jornada_proxy"] = (
    df_feat["id_cpf_cnpj"].astype("string").fillna("MISSING")
    + "__REV_" + df_feat["cod_rev"].astype("string").fillna("MISSING")
    + "__DT_" + df_feat["fsim_data_simulacao"].astype("string").fillna("MISSING")
)

df_feat = df_feat.sort_values(["id_jornada_proxy", "dt_simulacao", "id_chave_execucao"]).copy()

# Ordem e quantidade na jornada
df_feat["fsim_jorn_ordem_simulacao_ate_agora"] = (
    df_feat.groupby("id_jornada_proxy").cumcount() + 1
).astype("int32")

df_feat["fsim_jorn_is_primeira_simulacao"] = (
    df_feat["fsim_jorn_ordem_simulacao_ate_agora"] == 1
).astype("int8")

df_feat["fsim_jorn_is_terceira_ou_mais_simulacao"] = (
    df_feat["fsim_jorn_ordem_simulacao_ate_agora"] >= 3
).astype("int8")

# Taxa anterior na jornada
df_feat["fsim_jorn_tx_anterior"] = (
    df_feat.groupby("id_jornada_proxy")["tx_simul"].shift(1)
)

df_feat["fsim_jorn_tx_primeira"] = (
    df_feat.groupby("id_jornada_proxy")["tx_simul"].transform("first")
)

# Estatísticas até agora, incluindo a simulação atual
df_feat["fsim_jorn_tx_min_ate_agora"] = (
    df_feat.groupby("id_jornada_proxy")["tx_simul"].cummin()
)

df_feat["fsim_jorn_tx_max_ate_agora"] = (
    df_feat.groupby("id_jornada_proxy")["tx_simul"].cummax()
)

df_feat["fsim_jorn_tx_media_ate_agora"] = (
    df_feat.groupby("id_jornada_proxy")["tx_simul"]
    .expanding()
    .mean()
    .reset_index(level=0, drop=True)
)

df_feat["fsim_jorn_delta_tx_vs_anterior"] = (
    df_feat["tx_simul"] - df_feat["fsim_jorn_tx_anterior"]
)

df_feat["fsim_jorn_delta_tx_vs_primeira"] = (
    df_feat["tx_simul"] - df_feat["fsim_jorn_tx_primeira"]
)

df_feat["fsim_jorn_tx_caiu_vs_anterior"] = (
    df_feat["fsim_jorn_delta_tx_vs_anterior"] < 0
).astype("int8")

df_feat["fsim_jorn_is_menor_tx_ate_agora"] = (
    df_feat["tx_simul"] <= df_feat["fsim_jorn_tx_min_ate_agora"]
).astype("int8")

df_feat["fsim_jorn_amplitude_tx_ate_agora"] = (
    df_feat["fsim_jorn_tx_max_ate_agora"] - df_feat["fsim_jorn_tx_min_ate_agora"]
)

# Mudanças de condições versus simulação anterior
for col in ["num_prazo", "vl_entrada", "pct_entrada", "tp_politica", "tp_teste_elasticidade"]:
    prev_col = f"fsim_jorn_{col}_anterior"
    changed_col = f"fsim_jorn_mudou_{col}_vs_anterior"
    df_feat[prev_col] = df_feat.groupby("id_jornada_proxy")[col].shift(1)
    df_feat[changed_col] = (
        df_feat[col].astype("string") != df_feat[prev_col].astype("string")
    ).astype("int8")
    df_feat.loc[df_feat[prev_col].isna(), changed_col] = 0

df_feat[[
    "id_jornada_proxy",
    "dt_simulacao",
    "tx_simul",
    "fsim_jorn_ordem_simulacao_ate_agora",
    "fsim_jorn_delta_tx_vs_anterior",
    "fsim_jorn_tx_caiu_vs_anterior",
    "fsim_jorn_is_menor_tx_ate_agora",
]].head(15)


## 8. Features de estatísticas históricas de taxa por segmento

Aqui criamos features de preço relativo usando apenas o passado dentro do segmento.

Exemplo de segmento:
```text
tp_gh + id_cluster + fx_entrada + fx_ano_mod_veic
```

Para cada simulação, calculamos:
- média histórica da taxa no segmento antes daquela linha;
- quantidade histórica de simulações no segmento;
- diferença entre taxa atual e média histórica.

> Esta versão não usa target encoding com `fl_proposta`. Isso fica para uma segunda feature store, com validação temporal mais rigorosa.


In [ ]:
segment_cols = ["tp_gh", "id_cluster", "fx_entrada", "fx_ano_mod_veic"]

df_feat = df_feat.sort_values(segment_cols + ["dt_simulacao", "id_chave_execucao"]).copy()

# Quantidade histórica de simulações no segmento antes da linha atual
df_feat["fsim_seg_qtd_simulacoes_passadas"] = (
    df_feat.groupby(segment_cols).cumcount()
).astype("int32")

# Média histórica da taxa usando apenas linhas anteriores do segmento
df_feat["fsim_seg_tx_media_passada"] = (
    df_feat.groupby(segment_cols)["tx_simul"]
    .transform(lambda s: s.shift(1).expanding().mean())
)

df_feat["fsim_seg_tx_std_passada"] = (
    df_feat.groupby(segment_cols)["tx_simul"]
    .transform(lambda s: s.shift(1).expanding().std())
)

df_feat["fsim_seg_vl_financiado_media_passada"] = (
    df_feat.groupby(segment_cols)["vl_financiado"]
    .transform(lambda s: s.shift(1).expanding().mean())
)

df_feat["fsim_tx_vs_media_passada_segmento"] = (
    df_feat["tx_simul"] - df_feat["fsim_seg_tx_media_passada"]
)

df_feat["fsim_tx_ratio_media_passada_segmento"] = safe_divide(
    df_feat["tx_simul"],
    df_feat["fsim_seg_tx_media_passada"]
)

df_feat["fsim_is_tx_abaixo_media_passada_segmento"] = (
    df_feat["tx_simul"] < df_feat["fsim_seg_tx_media_passada"]
).astype("int8")

df_feat[[
    "tp_gh",
    "id_cluster",
    "fx_entrada",
    "fx_ano_mod_veic",
    "tx_simul",
    "fsim_seg_qtd_simulacoes_passadas",
    "fsim_seg_tx_media_passada",
    "fsim_tx_vs_media_passada_segmento",
]].head(15)


## 9. Seleção final de colunas da Feature Store

A feature store mantém:
- chaves;
- data de evento;
- target;
- features com prefixo `fsim_`;
- algumas variáveis base seguras da simulação.

Os identificadores não devem ser usados diretamente no modelo, mas são úteis para joins, auditoria e criação de features futuras.


In [ ]:
key_cols = [
    "id_chave_execucao",
    "id_pricing",
    "id_cpf_cnpj",
    "id_jornada_proxy",
    "dt_simulacao",
    "periodo_arquivo",
]

target_cols = ["target_proposta"]

base_safe_cols = [
    "id_cluster",
    "tp_politica",
    "tp_teste_elasticidade",
    "num_prazo",
    "cod_rev",
    "vl_financiado",
    "vl_entrada",
    "pct_entrada",
    "tp_gh",
    "fl_zero",
    "num_ano_mod_veic",
    "tx_simul",
    "tp_retorno_comissao",
    "fx_ano_mod_veic",
    "fx_entrada",
]

feature_cols = sorted([c for c in df_feat.columns if c.startswith("fsim_")])

fs_simulacoes = df_feat[key_cols + target_cols + base_safe_cols + feature_cols].copy()

print("Shape Feature Store:", fs_simulacoes.shape)
print("Qtd features derivadas:", len(feature_cols))
fs_simulacoes.head()


## 10. Split temporal simples

Sugestão inicial considerando períodos disponíveis entre 202511 e 202606:

- treino: até 202603;
- validação: 202604;
- teste/OOT: 202605 em diante.

Ajuste conforme a distribuição real dos seus dados.


In [ ]:
fs_simulacoes["ano_mes"] = pd.to_datetime(fs_simulacoes["dt_simulacao"]).dt.strftime("%Y%m")

fs_simulacoes["split"] = np.select(
    [
        fs_simulacoes["ano_mes"].between("202511", "202603"),
        fs_simulacoes["ano_mes"].eq("202604"),
        fs_simulacoes["ano_mes"].between("202605", "202606"),
    ],
    [
        "train",
        "valid",
        "test",
    ],
    default="out"
)

fs_simulacoes[["split", "target_proposta"]].value_counts(dropna=False).sort_index()


## 11. Auditorias rápidas anti-leakage

Procure por:
- features preenchidas somente quando `target_proposta = 1`;
- colunas com correlação suspeita;
- features com taxa de nulos muito diferente entre target 0 e 1.


In [ ]:
# Nulos por target nas features derivadas
null_by_target = (
    fs_simulacoes
    .groupby("target_proposta")[feature_cols]
    .apply(lambda x: x.isna().mean())
    .T
    .rename(columns={0: "null_rate_target_0", 1: "null_rate_target_1"})
)

null_by_target["abs_diff"] = (
    null_by_target["null_rate_target_1"] - null_by_target["null_rate_target_0"]
).abs()

null_by_target.sort_values("abs_diff", ascending=False).head(30)


In [ ]:
# Checagem de colunas suspeitas no resultado final
suspect_final_cols = [
    c for c in fs_simulacoes.columns
    if (
        c.endswith("_pond")
        or c in ["fl_aprovado", "fl_contrato", "id_contrato", "dt_contrato"]
    )
]

print("Colunas suspeitas na feature store final:", suspect_final_cols)
assert len(suspect_final_cols) == 0, "Feature Store contém colunas suspeitas de leakage."


## 12. Salvamento

A saída principal é:

```text
data/features/fs_simulacoes_proposta.parquet
```


In [ ]:
OUTPUT_PATH = FEATURE_DIR / "fs_simulacoes_proposta.parquet"
fs_simulacoes.to_parquet(OUTPUT_PATH, index=False)

print(f"Feature Store salva em: {OUTPUT_PATH}")
print(fs_simulacoes.shape)


## 13. Próximos passos sugeridos

1. Treinar um baseline com `base_safe_cols + fsim_*`.
2. Não usar `id_cpf_cnpj`, `id_pricing`, `id_chave_execucao` como features diretas.
3. Avaliar PR AUC, ROC AUC, Recall@K, Precision@K e Lift@K.
4. Depois criar uma segunda feature store com target encoding temporal e conversão histórica por revenda/cluster/política.
5. Validar se as features de quantidade de simulações realmente ajudam a encontrar o ponto ótimo de simulações por cliente/jornada.
